# Module SC7 — Multi-Backbone **Fusion** (feature-level combine)

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

Trains one model whose body is several backbones' pooled features concatenated → a fusion MLP → the 4 heads. By default it fuses the **per-head winners** chosen in Module SC6 (`fusion_backbones.json`); otherwise `T.FUSION_BACKBONES`. With `freeze=True` only the fusion+heads train (cheap).

**Inputs:** `thesis-common-10c`, image dataset, Module-SC1 output, the **SC2–SC5 outputs** (their `weights_<bb>.pt` — so fusion uses fine-tuned features), and (optional) Module-SC6 output (`fusion_backbones.json`).

**Outputs:** `logits_fusion.npz`, `weights_fusion.pt`, `metrics_fusion.json`, `history_fusion.csv` — attach to Module SC6 to compare.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

import torch
fb = T.find_one('fusion_backbones.json', required=False)
FUSE = json.load(open(fb)) if fb else T.FUSION_BACKBONES
if len(FUSE) < 2: FUSE = T.FUSION_BACKBONES          # need >=2 to fuse
# Load the FINE-TUNED backbones from SC2-SC5 (attach their outputs) -> fuse adapted features, not ImageNet.
WP = {}
for nm in FUSE:
    p = T.find_one(f'weights_{nm}.pt', required=False)
    if p: WP[nm] = p
print('fusing:', FUSE, '| fine-tuned weights found:', list(WP), '| freeze:', T.FUSION_FREEZE)
if not WP: print('  WARNING: no weights_*.pt attached -> fusing ImageNet features (weaker). Attach SC2-SC5 outputs.')
ROOT = T.find_image_root()
df_tr, df_va, df_te = T.load_manifests(ROOT); T.check_leakage(df_tr, df_va, df_te)
model, hist = T.train_fusion(FUSE, df_tr, df_va, df_te, weight_paths=WP)

### Evaluate, cache, rich metrics

In [ ]:

_, va, te = T.make_loaders(df_tr, df_va, df_te)
val = T.evaluate_logits(model, va); test = T.evaluate_logits(model, te)
T.save_logits('logits_fusion.npz', val, test)
torch.save(model.state_dict(), 'weights_fusion.pt'); hist.to_csv('history_fusion.csv', index=False)
rep = T.full_report(*test); json.dump(rep, open('metrics_fusion.json','w'), indent=2)
s = rep['scalars']
print('FUSION test  material:', s['material'], '\n             route   :', s['route'],
      '\n             threat  :', s['threat'], '\n             condition:', s['condition'])

### Compare fusion vs the single backbones (if their metrics_*.json are attached)

In [ ]:

import pandas as pd
rows = {'fusion': {'material_f1':rep['scalars']['material']['macro_f1'],
                   'route_f1':rep['scalars']['route']['macro_f1'],
                   'threat_qwk':rep['scalars']['threat']['qwk'],
                   'condition_auroc':rep['scalars']['condition']['auroc']}}
for nm in T.BACKBONES:
    p = T.find_one(f'metrics_{nm}.json', required=False)
    if p:
        r = json.load(open(p))['scalars']
        rows[nm] = {'material_f1':r['material']['macro_f1'],'route_f1':r['route']['macro_f1'],
                    'threat_qwk':r['threat']['qwk'],'condition_auroc':r['condition']['auroc']}
print(pd.DataFrame(rows).T.round(3))